# 1. Contexto del problema y dataset

## 1.1 Motivación

La **volatilidad** de un activo financiero — la magnitud típica de sus
fluctuaciones de precio — es una de las variables más importantes en
finanzas cuantitativas. Modelarla bien permite calibrar precios de
opciones, dimensionar posiciones, calcular *Value-at-Risk* y diseñar
estrategias de cobertura. A diferencia del retorno medio, que en
mercados eficientes es difícil de pronosticar, la volatilidad exhibe
**clustering**: períodos de alta volatilidad tienden a ser seguidos
por más alta volatilidad, y lo mismo en regímenes de calma. Este hecho
estilizado, documentado desde {cite}`engle1982autoregressive` y
formalizado en {cite}`bollerslev1986generalized`, da base predictiva
al problema.

## 1.2 Elección del activo

Trabajamos sobre **Intel Corporation (INTC)**, listado en NASDAQ. La
elección se justifica por tres razones:

1. **Cobertura temporal larga.** El histórico OHLCV disponible cubre
 más de 30 años, ofreciendo regímenes diversos (burbuja .com,
 crisis 2008, COVID-19, ciclo del semiconductor).
2. **Liquidez alta.** Reduce el ruido idiosincrático y hace que la
 volatilidad observada sea genuina y no producto de microestructura.
3. **Sector relevante.** El sector tecnológico tiene episodios de
 estrés y cambios estructurales claramente identificables, útiles
 para evaluar robustez de modelos a cambios de régimen.

## 1.3 Problema

Para cada día de trading $t$ disponemos del estado del mercado hasta
ese momento (precios apertura, máximo, mínimo, cierre y volumen, más
todas las transformaciones causales que de ahí se deriven). Queremos
predecir, para el día siguiente $t+1$:

- **Tarea de regresión:** la volatilidad realizada
 $\sigma_{t+1} = \mathrm{std}(\{r_{t-w+2}, \dots, r_{t+1}\})$,
 donde $r_s = \log(p_s / p_{s-1})$ y $w$ es la ventana causal
 (configuración: $w=5$).
- **Tarea de clasificación binaria:** el régimen
 $\rho_{t+1} \in \{0, 1\}$ — bajo o alto — definido por
 $\rho_{t+1} = \mathbb{1}\{\sigma_{t+1} > \tau\}$, donde $\tau$
 es la **mediana de $\sigma$ calculada únicamente sobre el conjunto
 de entrenamiento**.

## 1.4 Por qué horizonte de un día

El proyecto académico original de este equipo trabajó con horizontes
de 7, 14 y 21 días. En esa configuración el $R^2$ sobre test fue
negativo en todos los modelos evaluados — desde regresión lineal
hasta LSTM — síntoma de que la varianza condicional de la volatilidad
agregada a esos horizontes es tan baja en el período de prueba que
predecir la media histórica gana sistemáticamente.

Al fijar el horizonte en un día, recuperamos el efecto de clustering
de corto plazo, donde la dependencia serial de la volatilidad es
fuerte y predecible. Esta es la corrección metodológica central del
nuevo planteamiento.

## 1.5 Variables del dataset

El dataset crudo es la serie diaria OHLCV de INTC:

| Columna | Descripción |
|---|---|
| `date` | Fecha de la sesión |
| `open` | Precio de apertura |
| `high` | Máximo intradiario |
| `low` | Mínimo intradiario |
| `close` | Precio de cierre |
| `volume` | Volumen negociado (acciones) |

A partir de estas seis variables construimos, en el capítulo 3, ~30
features causales que capturan retornos rezagados, volatilidad
rolling, componentes HAR-RV (Corsi 2009), momentum, rangos de precio,
indicadores técnicos y variables de calendario.

## 1.6 Conexión con el capítulo 2

En el siguiente capítulo descargamos los datos (con `yfinance` como
fallback automático), aplicamos limpieza básica, verificamos integridad
y producimos el primer análisis exploratorio orientado a justificar
la elección del problema y a documentar los hechos estilizados de
INTC en los que descansará todo el modelado posterior.